## Boosting

#### CatBoost

In [4]:
# --- 1. Importer nødvendige pakker ---
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import numpy as np

# --- 2. Les inn dimensjonsredusert data ---
df = pd.read_csv('cyberbullying_datasets/course_dataset/cb_chi_compressed.csv.gz')
print("✅ Data loaded:", df.shape)

X = df.drop("bullying", axis=1)
y = df["bullying"]

# --- 3. Split i trenings- og testsett ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 4. Opprett CatBoost-modellen ---
cat_model = CatBoostClassifier(
    iterations=500,          # hvor mange trær som bygges (øker nøyaktighet, men også tid)
    depth=6,                 # tre-dybde (balanse mellom kompleksitet og generalisering)
    learning_rate=0.05,      # hvor mye hvert tre "lærer" per steg
    loss_function='Logloss', # binær klassifisering
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100              # skriver status for hver 100 iterasjon
)

# --- 5. Tren modellen ---
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))

# --- 6. Evaluer på testsett ---
y_pred = cat_model.predict(X_test)

✅ Data loaded: (11069, 5001)
0:	learn: 0.5725579	test: 0.5709124	best: 0.5709124 (0)	total: 280ms	remaining: 2m 19s
100:	learn: 0.6965556	test: 0.6874435	best: 0.6874435 (99)	total: 3.47s	remaining: 13.7s
200:	learn: 0.7245624	test: 0.7073171	best: 0.7073171 (195)	total: 6.64s	remaining: 9.88s
300:	learn: 0.7486166	test: 0.7104788	best: 0.7104788 (285)	total: 9.57s	remaining: 6.33s
400:	learn: 0.7670243	test: 0.7149955	best: 0.7168022 (344)	total: 12.6s	remaining: 3.12s
499:	learn: 0.7817053	test: 0.7109304	best: 0.7168022 (344)	total: 15.8s	remaining: 0us

bestTest = 0.716802168
bestIteration = 344

Shrink model to first 345 iterations.


In [5]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7168021680216802

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.83      0.75      1105
           1       0.78      0.60      0.68      1109

    accuracy                           0.72      2214
   macro avg       0.73      0.72      0.71      2214
weighted avg       0.73      0.72      0.71      2214



#### XGboost

In [ ]:
from xgboost import XGBClassifier

XGBClassifier(n_estimators=800,learning_rate=0.01,max_depth=8,subsample=0.8,colsample_bytree=0.8,eval_metric='logloss')

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

## Chi-kvadrat data
df = pd.read_csv('cyberbullying_datasets/course_dataset/cb_chi_compressed.csv.gz')
print(df.shape)
df


X = df.drop(columns=["bullying"])
y = df["bullying"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)


xgb_model = XGBClassifier(
    n_estimators=800,       # number of boosting rounds
    learning_rate=0.01,     # smaller = slower but better generalization
    max_depth=8,            # depth of each tree
    subsample=0.8,          # randomly sample training data (helps avoid overfitting)
    colsample_bytree=0.8,   # randomly sample features
    random_state=0,
    use_label_encoder=False,
    eval_metric='logloss'
)

# # K-fold cross validation 
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# scores = []

# for train_idx, test_idx in cv.split(X, y):
#     X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
#     y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

#     xgb_model.fit(X_train_cv, y_train_cv)
#     y_pred_cv = xgb_model.predict(X_test_cv)
#     scores.append(accuracy_score(y_test_cv, y_pred_cv))

# print(f"Cross-validation accuracies: {scores}")
# print(f"Mean CV accuracy: {np.mean(scores):.4f}")

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


(11069, 5001)


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [02:06:08] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Accuracy: 0.7000903342366757

Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.84      0.74      1105
           1       0.78      0.56      0.65      1109

    accuracy                           0.70      2214
   macro avg       0.72      0.70      0.69      2214
weighted avg       0.72      0.70      0.69      2214



In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

## Chi-kvadrat data
df = pd.read_csv('cyberbullying_datasets/course_dataset/cb_chi_compressed.csv.gz')
print("Shape:", df.shape)

X = df.drop(columns=["bullying"])
y = df["bullying"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=0
)

xgb_model = XGBClassifier(
    n_estimators=800,       # number of boosting rounds
    learning_rate=0.01,    # smaller = slower but better generalization
    max_depth=8,           # depth of each tree
    subsample=0.8,         # randomly sample training data (helps avoid overfitting)
    colsample_bytree=0.8,  # randomly sample features
    random_state=0,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Tren modellen
xgb_model.fit(X_train, y_train)

# --- 1) SCORE PÅ TRAIN OG TEST ---
train_pred = xgb_model.predict(X_train)
test_pred  = xgb_model.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc  = accuracy_score(y_test,  test_pred)

print(f"\nTrain accuracy: {train_acc:.4f}")
print(f"Test  accuracy: {test_acc:.4f}")
print(f"Generalization gap (train - test): {train_acc - test_acc:+.4f}")

# Enkel indikator på mulig overfitting
gap = train_acc - test_acc
if gap > 0.10:
    print("⚠️  Mulig sterk overfitting (gap > 0.10). Vurder å regularisere mer.")
elif gap > 0.05:
    print("⚠️  Tegn til overfitting (gap > 0.05). Vurder tiltak.")
else:
    print("✅ Ingen tydelige tegn til overfitting basert på accuracy-gapet.")

# --- 2) RAPPORTER / MATRISER ---
print("\nClassification Report (TEST):\n", classification_report(y_test, test_pred))
print("Confusion Matrix (TEST):\n", confusion_matrix(y_test, test_pred))

# (Valgfritt) Kommenter inn for å se treningsrapport
# print("\nClassification Report (TRAIN):\n", classification_report(y_train, train_pred))
# print("Confusion Matrix (TRAIN):\n", confusion_matrix(y_train, train_pred))

Shape: (11069, 5001)


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [02:05:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Train accuracy: 0.7377
Test  accuracy: 0.7001
Generalization gap (train - test): +0.0376
✅ Ingen tydelige tegn til overfitting basert på accuracy-gapet.

Classification Report (TEST):
               precision    recall  f1-score   support

           0       0.66      0.84      0.74      1105
           1       0.78      0.56      0.65      1109

    accuracy                           0.70      2214
   macro avg       0.72      0.70      0.69      2214
weighted avg       0.72      0.70      0.69      2214

Confusion Matrix (TEST):
 [[926 179]
 [485 624]]


#### SE BORT IFRA UNDER

In [1]:
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 300, 500],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

grid = GridSearchCV(xgb_model, params, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)


NameError: name 'xgb_model' is not defined

In [10]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for n in param_grid["n_estimators"]:
    for d in param_grid["max_depth"]:
        for lr in param_grid["learning_rate"]:
            for sub in param_grid["subsample"]:
                xgb_model = XGBClassifier(
                    n_estimators=n,
                    max_depth=d,
                    learning_rate=lr,
                    subsample=sub,
                    colsample_bytree=0.8,
                    random_state=42,
                    use_label_encoder=False,
                    eval_metric="logloss"
                )


                xgb_model.fit(X_train, y_train)
                y_pred = xgb_model.predict(X_test)

                results.append({
                    "n_estimators": n,
                    "max_depth": d,
                    "learning_rate": lr,
                    "subsample": sub,
                })
                print(f"n={n}, depth={d}, lr={lr}, subsample={sub}")

results_df = pd.DataFrame(results)
best_params = results_df.loc[results_df['mean_f1'].idxmax()]
print("\nBest parameters:")
print(best_params)


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [15:19:38] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=100, depth=3, lr=0.01, subsample=0.8


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [15:20:11] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=100, depth=3, lr=0.01, subsample=1.0


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [15:20:44] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


...

C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [23:05:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=300, depth=7, lr=0.1, subsample=0.8


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [01:15:38] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=300, depth=7, lr=0.1, subsample=1.0


KeyError: 'mean_f1'

## Bagging

#### SVM BAGGING

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC

BaggingClassifier(
    estimator=SVC(kernel="rbf", C=1,gamma="scale", random_state=0),
    n_estimators=50,
    random_state=0
)

In [10]:
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score


df = pd.read_csv('cyberbullying_datasets/course_dataset/cb_chi_compressed.csv.gz')


X = df.drop(columns=["bullying"])
y = df["bullying"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

bagged_svm = BaggingClassifier(
    estimator=SVC(kernel="rbf", C=1,gamma="scale", random_state=0),
    n_estimators=50,
    random_state=0
).fit(X_train, y_train)

y_pred = bagged_svm.predict(X_test)

In [11]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7655826558265583

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.85      0.78      1105
           1       0.82      0.69      0.75      1109

    accuracy                           0.77      2214
   macro avg       0.77      0.77      0.76      2214
weighted avg       0.77      0.77      0.76      2214



In [7]:
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


df_pca = pd.read_csv("cyberbullying_datasets/course_dataset/cb_chi_compressed.csv.gz")


X = df_pca.drop(columns=["bullying"])
y = df_pca["bullying"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

bagged_svm = BaggingClassifier(
    estimator=LogisticRegression(max_iter=10000, C=15.0, penalty='l2', solver="lbfgs", random_state=0),
    n_estimators=100,
    random_state=0
).fit(X_train, y_train)

y_pred = bagged_svm.predict(X_test)
y_pred_train = bagged_svm.predict(X_train)

In [8]:
print("Accuracy ts:", accuracy_score(y_test, y_pred))
print("Accuracy tr:", accuracy_score(y_train, y_pred_train))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy ts: 0.7726588376994881
Accuracy tr: 0.8546721734641197

Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.82      0.78      1657
           1       0.80      0.73      0.76      1664

    accuracy                           0.77      3321
   macro avg       0.77      0.77      0.77      3321
weighted avg       0.78      0.77      0.77      3321

